In [ ]:
!nvidia-smi

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%pip install -q "transformers>=4.51" safetensors einops packaging ninja

In [ ]:
import platform
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

In [ ]:
!nvidia-smi
!nvcc --version
!pip show flash-attn
!pip show torch

In [ ]:
import os

os.environ["MAX_JOBS"] = "4"

print("MAX_JOBS:", os.environ["MAX_JOBS"])

In [ ]:
%pip install flash-attn --no-build-isolation

In [ ]:
import flash_attn
import torch
import transformers

print("flash-attn:", flash_attn.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

from google.colab import userdata
from huggingface_hub import HfApi, login

raw_token = userdata.get("HF_TOKEN")

if not raw_token:
    raise RuntimeError(
        "HF_TOKEN is unavailable in Colab Secrets."
    )

token_lines = [
    line.strip()
    for line in raw_token.splitlines()
    if line.strip()
]

if len(token_lines) != 1:
    raise RuntimeError(
        "HF_TOKEN must contain exactly "
        "one non-empty line."
    )

HF_TOKEN = token_lines[0]

if not HF_TOKEN.startswith("hf_"):
    raise RuntimeError(
        "HF_TOKEN has an unexpected format."
    )

if any(
    character.isspace()
    for character in HF_TOKEN
):
    raise RuntimeError(
        "HF_TOKEN contains whitespace."
    )

login(
    token=HF_TOKEN,
    add_to_git_credential=False,
)

api = HfApi(token=HF_TOKEN)
identity = api.whoami()

DATASET_REPO = (
    "AliothMe/lung-fusion-tile-shards"
)
DATASET_REVISION = (
    "899e7b2c754b8154887f4e818b9482dfbda0c9bc"
)

EMBEDDING_REPO = (
    "AliothMe/lung-fusion-embeddings"
)
VIRCHOW2_CACHE_REVISION = (
    "1f88a0e0b24d63952af31816de4d2d9dca7441a0"
)

PRISM2_REPO = "paige-ai/Prism2"

WORK_DIR = Path(
    "/content/lung-fusion-agent"
)
PRISM2_INPUT_DIR = (
    WORK_DIR / "prism2_inputs"
)
PRISM2_OUTPUT_DIR = (
    WORK_DIR / "prism2_outputs"
)

PRISM2_INPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
PRISM2_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Authenticated user:", identity["name"])
print("Input dataset:", DATASET_REPO)
print(
    "Dataset revision:",
    DATASET_REVISION,
)
print(
    "Embedding repository:",
    EMBEDDING_REPO,
)
print(
    "Virchow2 cache revision:",
    VIRCHOW2_CACHE_REVISION,
)
print("Prism2 repository:", PRISM2_REPO)

In [ ]:
required_names = [
    "HF_TOKEN",
    "api",
    "DATASET_REPO",
    "DATASET_REVISION",
    "EMBEDDING_REPO",
    "VIRCHOW2_CACHE_REVISION",
    "PRISM2_REPO",
    "PRISM2_INPUT_DIR",
    "PRISM2_OUTPUT_DIR",
]

for name in required_names:
    print(
        f"{name}:",
        "ready" if name in globals() else "MISSING",
    )

In [ ]:
prism2_info = api.model_info(
    PRISM2_REPO
)

PRISM2_REVISION = prism2_info.sha

print(
    "Prism2 revision:",
    PRISM2_REVISION,
)

In [ ]:
import gc

from transformers import AutoModel, AutoProcessor

gc.collect()
torch.cuda.empty_cache()

print("Loading Prism2 model...")

processor = AutoProcessor.from_pretrained(
    PRISM2_REPO,
    revision=PRISM2_REVISION,
    trust_remote_code=True,
    token=HF_TOKEN,
)

model = AutoModel.from_pretrained(
    PRISM2_REPO,
    revision=PRISM2_REVISION,
    trust_remote_code=True,
    torch_dtype="auto",
    token=HF_TOKEN,
)

model = model.cuda().eval()

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Prism2 loaded.")
print("Parameters:", f"{parameter_count:,}")
print(
    "Parameters in billions:",
    round(parameter_count / 1_000_000_000, 2),
)
print(
    "Model dtype:",
    next(model.parameters()).dtype,
)
print(
    "GPU allocated:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2,
    ),
    "GiB",
)

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

tile_index_path = hf_hub_download(
    repo_id=DATASET_REPO,
    filename="manifests/tile_to_shard.csv.gz",
    repo_type="dataset",
    revision=DATASET_REVISION,
    token=HF_TOKEN,
)

tile_index = pd.read_csv(
    tile_index_path
)

print("Tile index loaded.")
print("Rows:", len(tile_index))
print(
    "Unique tile IDs:",
    tile_index["tile_id"].nunique(),
)
print(
    "Unique WSIs:",
    tile_index["wsi_id"].nunique(),
)
print(
    "Unique patients:",
    tile_index["patient_id"].nunique(),
)
print(tile_index.head())

In [ ]:
required_names = [
    "HF_TOKEN",
    "api",
    "model",
    "processor",
    "DATASET_REPO",
    "DATASET_REVISION",
    "EMBEDDING_REPO",
    "VIRCHOW2_CACHE_REVISION",
    "PRISM2_REPO",
    "PRISM2_REVISION",
    "tile_index",
]

for name in required_names:
    print(
        f"{name}:",
        "ready" if name in globals() else "MISSING",
    )

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

tile_index_path = hf_hub_download(
    repo_id=DATASET_REPO,
    filename="manifests/tile_to_shard.csv.gz",
    repo_type="dataset",
    revision=DATASET_REVISION,
    token=HF_TOKEN,
)

tile_index = pd.read_csv(tile_index_path)

print("Rows:", len(tile_index))
print(
    "Unique tile IDs:",
    tile_index["tile_id"].nunique(),
)
print(
    "Unique WSIs:",
    tile_index["wsi_id"].nunique(),
)

In [ ]:
from pathlib import Path

import numpy as np

VIRCHOW2_CACHE_REVISION = (
    "1f88a0e0b24d63952af31816de4d2d9dca7441a0"
)

TARGET_WSI_ID = tile_index.iloc[0]["wsi_id"]

target_rows = tile_index.loc[
    tile_index["wsi_id"] == TARGET_WSI_ID
].copy()

target_rows = target_rows.reset_index(
    drop=True
)

print("Target WSI:", TARGET_WSI_ID)
print("Expected tiles:", len(target_rows))
print(
    "Required shards:",
    target_rows["shard_name"].unique(),
)

In [ ]:
class_embedding_by_tile = {}

for shard_name in target_rows[
    "shard_name"
].unique():
    shard_index = int(
        Path(shard_name).stem.split("-")[-1]
    )

    virchow_filename = (
        f"virchow2/"
        f"virchow2-{shard_index:05d}.npz"
    )

    virchow_path = hf_hub_download(
        repo_id=EMBEDDING_REPO,
        filename=virchow_filename,
        repo_type="dataset",
        revision=VIRCHOW2_CACHE_REVISION,
        token=HF_TOKEN,
    )

    with np.load(
        virchow_path,
        allow_pickle=False,
    ) as cache:
        cached_ids = (
            cache["tile_ids"].astype(str)
        )
        cached_embeddings = (
            cache["embeddings"]
        )

        if cached_embeddings.shape[1] != 2560:
            raise RuntimeError(
                "Unexpected Virchow2 dimension: "
                f"{cached_embeddings.shape}"
            )

        # Prism2 only needs the first 1280
        # dimensions: Virchow2 class token.
        class_embeddings = (
            cached_embeddings[:, :1280]
            .astype(np.float32)
        )

        for tile_id, embedding in zip(
            cached_ids,
            class_embeddings,
            strict=True,
        ):
            class_embedding_by_tile[
                tile_id
            ] = embedding

ordered_tile_ids = (
    target_rows["tile_id"]
    .astype(str)
    .tolist()
)

missing_ids = [
    tile_id
    for tile_id in ordered_tile_ids
    if tile_id
    not in class_embedding_by_tile
]

if missing_ids:
    raise RuntimeError(
        f"Missing {len(missing_ids)} tiles"
    )

slide_array = np.stack(
    [
        class_embedding_by_tile[tile_id]
        for tile_id in ordered_tile_ids
    ]
)

slide_tensor = torch.from_numpy(
    slide_array
)

print("Slide:", TARGET_WSI_ID)
print("Slide tensor:", slide_tensor.shape)
print("Dtype:", slide_tensor.dtype)
print(
    "All finite:",
    bool(torch.isfinite(slide_tensor).all()),
)

In [ ]:
import time

batch = processor(
    tile_embeddings=[slide_tensor]
).to("cuda")

print(
    "Processor tile embeddings:",
    batch["tile_embeddings"].shape,
)
print(
    "Attention mask:",
    batch["attention_mask"].shape,
)

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start_time = time.perf_counter()

with torch.inference_mode(), torch.autocast(
    device_type="cuda",
    dtype=torch.bfloat16,
):
    prism2_base = (
        model.get_base_embedding(**batch)
    )

torch.cuda.synchronize()
elapsed_seconds = (
    time.perf_counter() - start_time
)

print(
    "Prism2 base shape:",
    tuple(prism2_base.shape),
)
print("Dtype:", prism2_base.dtype)
print(
    "All finite:",
    bool(torch.isfinite(prism2_base).all()),
)
print(
    "L2 norm:",
    round(
        float(
            prism2_base.float()
            .norm(dim=1)
            .mean()
        ),
        4,
    ),
)
print(
    "Elapsed seconds:",
    round(elapsed_seconds, 3),
)
print(
    "Peak GPU memory:",
    round(
        torch.cuda.max_memory_allocated()
        / 1024**3,
        2,
    ),
    "GiB",
)

In [ ]:
import gc

if "batch" in globals():
    del batch

if "prism2_base" in globals():
    del prism2_base

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2,
    ),
    "GiB",
)

In [ ]:
from collections import defaultdict

import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

tile_to_wsi = (
    tile_index
    .set_index("tile_id")["wsi_id"]
)

slide_parts = defaultdict(list)
all_loaded_tile_ids = set()

for shard_index in range(49):
    embedding_filename = (
        f"virchow2/"
        f"virchow2-{shard_index:05d}.npz"
    )

    embedding_path = hf_hub_download(
        repo_id=EMBEDDING_REPO,
        filename=embedding_filename,
        repo_type="dataset",
        revision=VIRCHOW2_CACHE_REVISION,
        token=HF_TOKEN,
    )

    with np.load(
        embedding_path,
        allow_pickle=False,
    ) as cache:
        cached_ids = (
            cache["tile_ids"]
            .astype(str)
        )
        cached_embeddings = (
            cache["embeddings"]
        )

        if cached_embeddings.shape[1] != 2560:
            raise RuntimeError(
                f"{embedding_filename}: "
                f"unexpected shape "
                f"{cached_embeddings.shape}"
            )

        overlap = (
            all_loaded_tile_ids
            & set(cached_ids.tolist())
        )

        if overlap:
            raise RuntimeError(
                f"{embedding_filename}: "
                f"{len(overlap)} duplicate IDs"
            )

        all_loaded_tile_ids.update(
            cached_ids.tolist()
        )

        try:
            cached_wsi_ids = (
                tile_to_wsi.loc[
                    cached_ids
                ].to_numpy()
            )
        except KeyError as error:
            raise RuntimeError(
                f"{embedding_filename}: "
                "contains an unknown tile ID"
            ) from error

        # Prism2 requires class token only:
        # first 1280 dimensions.
        class_embeddings = (
            cached_embeddings[:, :1280]
            .astype(np.float16)
        )

        for wsi_id in pd.unique(
            cached_wsi_ids
        ):
            mask = cached_wsi_ids == wsi_id

            slide_parts[wsi_id].append(
                class_embeddings[mask].copy()
            )

    print(
        f"[{shard_index + 1:02d}/49] "
        f"{embedding_filename} loaded"
    )

In [ ]:
slide_bags = {}

for wsi_id, parts in slide_parts.items():
    combined = np.concatenate(
        parts,
        axis=0,
    )

    slide_bags[wsi_id] = (
        torch.from_numpy(combined)
    )

expected_counts = (
    tile_index
    .groupby("wsi_id")
    .size()
    .to_dict()
)

if len(slide_bags) != 204:
    raise RuntimeError(
        f"Expected 204 slide bags, "
        f"found {len(slide_bags)}"
    )

if len(all_loaded_tile_ids) != 200488:
    raise RuntimeError(
        f"Expected 200488 tile IDs, "
        f"found {len(all_loaded_tile_ids)}"
    )

for wsi_id, slide_tensor in slide_bags.items():
    expected_count = expected_counts[wsi_id]

    if slide_tensor.shape != (
        expected_count,
        1280,
    ):
        raise RuntimeError(
            f"{wsi_id}: shape "
            f"{tuple(slide_tensor.shape)}, "
            f"expected "
            f"({expected_count}, 1280)"
        )

    if not torch.isfinite(
        slide_tensor
    ).all():
        raise RuntimeError(
            f"{wsi_id}: contains "
            "NaN or infinity"
        )

tile_counts = [
    tensor.shape[0]
    for tensor in slide_bags.values()
]

print("=== Slide-bag validation ===")
print("Slides:", len(slide_bags))
print(
    "Loaded unique tiles:",
    len(all_loaded_tile_ids),
)
print("Total bag rows:", sum(tile_counts))
print("Min tiles:", min(tile_counts))
print(
    "Median tiles:",
    float(np.median(tile_counts)),
)
print("Max tiles:", max(tile_counts))
print(
    "Input dimension:",
    {
        tensor.shape[1]
        for tensor in slide_bags.values()
    },
)
print(
    "Input dtypes:",
    {
        str(tensor.dtype)
        for tensor in slide_bags.values()
    },
)

In [ ]:
import statistics
import time

ordered_wsi_ids = sorted(
    slide_bags.keys(),
    key=lambda value: int(
        value.split("-")[-1]
    ),
)

benchmark_wsi_ids = ordered_wsi_ids[:8]


def benchmark_prism2(
    slide_batch_size: int,
    repeats: int = 2,
) -> dict[str, float]:
    elapsed_times = []

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    for _ in range(repeats):
        torch.cuda.synchronize()
        start_time = time.perf_counter()

        for start in range(
            0,
            len(benchmark_wsi_ids),
            slide_batch_size,
        ):
            batch_ids = benchmark_wsi_ids[
                start : start
                + slide_batch_size
            ]

            slides = [
                slide_bags[wsi_id]
                for wsi_id in batch_ids
            ]

            processed_batch = processor(
                tile_embeddings=slides
            ).to("cuda")

            with torch.inference_mode(), torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            ):
                output = (
                    model.get_base_embedding(
                        **processed_batch
                    )
                )

            if output.shape != (
                len(batch_ids),
                2560,
            ):
                raise RuntimeError(
                    f"Unexpected output shape: "
                    f"{tuple(output.shape)}"
                )

            del processed_batch
            del output

        torch.cuda.synchronize()

        elapsed_times.append(
            time.perf_counter() - start_time
        )

    median_seconds = statistics.median(
        elapsed_times
    )

    return {
        "batch_size": slide_batch_size,
        "slides_per_second": (
            len(benchmark_wsi_ids)
            / median_seconds
        ),
        "peak_memory_gib": (
            torch.cuda.max_memory_allocated()
            / 1024**3
        ),
    }


for tested_batch_size in [1, 2, 4]:
    result = benchmark_prism2(
        tested_batch_size
    )

    print(
        f"Batch {tested_batch_size} | "
        f"{result['slides_per_second']:.3f} "
        "slides/s | "
        f"{result['peak_memory_gib']:.2f} GiB"
    )

In [ ]:
import time

import numpy as np
import torch

SLIDE_BATCH_SIZE = 4
PRISM2_EMBEDDING_DIM = 2560

ordered_wsi_ids = sorted(
    slide_bags.keys(),
    key=lambda value: int(
        value.split("-")[-1]
    ),
)

all_prism2_embeddings = []

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

full_start_time = time.perf_counter()

for start in range(
    0,
    len(ordered_wsi_ids),
    SLIDE_BATCH_SIZE,
):
    batch_wsi_ids = ordered_wsi_ids[
        start : start + SLIDE_BATCH_SIZE
    ]

    input_slides = [
        slide_bags[wsi_id]
        for wsi_id in batch_wsi_ids
    ]

    processed_batch = processor(
        tile_embeddings=input_slides
    ).to("cuda")

    with torch.inference_mode(), torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):
        batch_embeddings = (
            model.get_base_embedding(
                **processed_batch
            )
        )

    expected_shape = (
        len(batch_wsi_ids),
        PRISM2_EMBEDDING_DIM,
    )

    if batch_embeddings.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected output shape "
            f"{tuple(batch_embeddings.shape)}; "
            f"expected {expected_shape}"
        )

    if not torch.isfinite(
        batch_embeddings
    ).all():
        raise RuntimeError(
            f"Non-finite output for "
            f"{batch_wsi_ids}"
        )

    all_prism2_embeddings.append(
        batch_embeddings
        .to(dtype=torch.float16)
        .cpu()
    )

    completed = min(
        start + SLIDE_BATCH_SIZE,
        len(ordered_wsi_ids),
    )

    if (
        completed % 20 == 0
        or completed == len(ordered_wsi_ids)
    ):
        print(
            f"Processed {completed}/"
            f"{len(ordered_wsi_ids)} slides"
        )

    del processed_batch
    del batch_embeddings

torch.cuda.synchronize()

full_elapsed_seconds = (
    time.perf_counter() - full_start_time
)

prism2_embeddings = torch.cat(
    all_prism2_embeddings,
    dim=0,
)

peak_memory_gib = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("=== Prism2 full extraction ===")
print(
    "Embedding shape:",
    tuple(prism2_embeddings.shape),
)
print(
    "Dtype:",
    prism2_embeddings.dtype,
)
print(
    "All finite:",
    bool(
        torch.isfinite(
            prism2_embeddings
        ).all()
    ),
)
print(
    "Unique WSI IDs:",
    len(set(ordered_wsi_ids)),
)
print(
    "Elapsed seconds:",
    round(full_elapsed_seconds, 3),
)
print(
    "Slides per second:",
    round(
        len(ordered_wsi_ids)
        / full_elapsed_seconds,
        3,
    ),
)
print(
    "Peak GPU memory:",
    round(peak_memory_gib, 2),
    "GiB",
)
print(
    "Mean L2 norm:",
    round(
        float(
            prism2_embeddings
            .float()
            .norm(dim=1)
            .mean()
        ),
        4,
    ),
)

In [ ]:
import pandas as pd

wsi_metadata = (
    tile_index
    .groupby("wsi_id", as_index=False)
    .agg(
        patient_id=("patient_id", "first"),
        patient_id_count=(
            "patient_id",
            "nunique",
        ),
        tile_count=("tile_id", "count"),
    )
)

if (
    wsi_metadata["patient_id_count"].max()
    != 1
):
    raise RuntimeError(
        "A WSI maps to multiple patient IDs."
    )

wsi_metadata = (
    wsi_metadata
    .set_index("wsi_id")
    .loc[ordered_wsi_ids]
    .reset_index()
)

prism2_manifest = pd.DataFrame(
    {
        "row_index": range(
            len(ordered_wsi_ids)
        ),
        "wsi_id": ordered_wsi_ids,
        "patient_id": (
            wsi_metadata["patient_id"]
            .tolist()
        ),
        "tile_count": (
            wsi_metadata["tile_count"]
            .astype(int)
            .tolist()
        ),
        "embedding_dimension": (
            PRISM2_EMBEDDING_DIM
        ),
        "embedding_dtype": "float16",
    }
)

print(prism2_manifest.head())
print()
print("Rows:", len(prism2_manifest))
print(
    "Unique WSIs:",
    prism2_manifest["wsi_id"].nunique(),
)
print(
    "Total input tiles:",
    prism2_manifest["tile_count"].sum(),
)
print(
    "Min tiles:",
    prism2_manifest["tile_count"].min(),
)
print(
    "Median tiles:",
    prism2_manifest[
        "tile_count"
    ].median(),
)
print(
    "Max tiles:",
    prism2_manifest["tile_count"].max(),
)

In [ ]:
import hashlib
import json
from pathlib import Path


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while chunk := file.read(1024 * 1024):
            digest.update(chunk)

    return digest.hexdigest()


embedding_output_path = (
    PRISM2_OUTPUT_DIR
    / "prism2_base_embeddings.npz"
)
manifest_output_path = (
    PRISM2_OUTPUT_DIR
    / "manifest.csv"
)
metadata_output_path = (
    PRISM2_OUTPUT_DIR
    / "run_metadata.json"
)

embedding_array = (
    prism2_embeddings.numpy()
)

np.savez_compressed(
    embedding_output_path,
    wsi_ids=np.asarray(
        ordered_wsi_ids
    ),
    embeddings=embedding_array,
    tile_counts=prism2_manifest[
        "tile_count"
    ].to_numpy(dtype=np.int32),
)

prism2_manifest.to_csv(
    manifest_output_path,
    index=False,
)

embedding_checksum = sha256_file(
    embedding_output_path
)

run_metadata = {
    "model_name": "Prism2",
    "model_repo": PRISM2_REPO,
    "model_revision": PRISM2_REVISION,
    "input_embedding_model": "Virchow2",
    "input_embedding_revision": (
        VIRCHOW2_CACHE_REVISION
    ),
    "input_feature": (
        "virchow2_class_token_only"
    ),
    "input_dimension": 1280,
    "input_tile_count": 200488,
    "slide_count": 204,
    "output_embedding": "base",
    "output_dimension": 2560,
    "output_dtype": "float16",
    "slide_batch_size": SLIDE_BATCH_SIZE,
    "elapsed_seconds": full_elapsed_seconds,
    "slides_per_second": (
        len(ordered_wsi_ids)
        / full_elapsed_seconds
    ),
    "peak_gpu_memory_gib": peak_memory_gib,
    "gpu": torch.cuda.get_device_name(0),
    "torch_version": torch.__version__,
    "transformers_version": (
        transformers.__version__
    ),
    "flash_attn_version": (
        flash_attn.__version__
    ),
    "embedding_file": (
        embedding_output_path.name
    ),
    "embedding_size_bytes": (
        embedding_output_path
        .stat()
        .st_size
    ),
    "embedding_sha256": (
        embedding_checksum
    ),
}

metadata_output_path.write_text(
    json.dumps(
        run_metadata,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

print("Saved:", embedding_output_path)
print("Saved:", manifest_output_path)
print("Saved:", metadata_output_path)
print(
    "Embedding file size:",
    round(
        embedding_output_path
        .stat()
        .st_size
        / 1024**2,
        2,
    ),
    "MiB",
)
print("SHA-256:", embedding_checksum)

In [ ]:
with np.load(
    embedding_output_path,
    allow_pickle=False,
) as cache:
    cached_wsi_ids = cache[
        "wsi_ids"
    ].astype(str)
    cached_embeddings = cache[
        "embeddings"
    ]
    cached_tile_counts = cache[
        "tile_counts"
    ]

    print(
        "WSI IDs:",
        cached_wsi_ids.shape,
    )
    print(
        "Embeddings:",
        cached_embeddings.shape,
    )
    print(
        "Tile counts:",
        cached_tile_counts.shape,
    )
    print(
        "Dtype:",
        cached_embeddings.dtype,
    )
    print(
        "All finite:",
        bool(
            np.isfinite(
                cached_embeddings
            ).all()
        ),
    )
    print(
        "Unique WSI IDs:",
        len(set(cached_wsi_ids)),
    )
    print(
        "Total tile count:",
        int(cached_tile_counts.sum()),
    )

In [ ]:
upload_result = api.upload_folder(
    repo_id=EMBEDDING_REPO,
    repo_type="dataset",
    folder_path=str(
        PRISM2_OUTPUT_DIR
    ),
    path_in_repo="prism2",
    commit_message=(
        "Add complete Prism2 base "
        "slide embeddings"
    ),
)

print("Upload complete:")
print(upload_result)

In [ ]:
final_repo_info = api.repo_info(
    EMBEDDING_REPO,
    repo_type="dataset",
)

PRISM2_CACHE_REVISION = (
    final_repo_info.sha
)

print(
    "Prism2 cache revision:",
    PRISM2_CACHE_REVISION,
)

In [ ]:
from huggingface_hub import hf_hub_download

PRISM2_CACHE_REVISION = (
    "6d979cd1156c4efd4b830f90f0f1c0da3a59c140"
)

remote_embedding_path = hf_hub_download(
    repo_id=EMBEDDING_REPO,
    filename=(
        "prism2/prism2_base_embeddings.npz"
    ),
    repo_type="dataset",
    revision=PRISM2_CACHE_REVISION,
    token=HF_TOKEN,
    force_download=True,
)

remote_manifest_path = hf_hub_download(
    repo_id=EMBEDDING_REPO,
    filename="prism2/manifest.csv",
    repo_type="dataset",
    revision=PRISM2_CACHE_REVISION,
    token=HF_TOKEN,
    force_download=True,
)

remote_metadata_path = hf_hub_download(
    repo_id=EMBEDDING_REPO,
    filename="prism2/run_metadata.json",
    repo_type="dataset",
    revision=PRISM2_CACHE_REVISION,
    token=HF_TOKEN,
    force_download=True,
)

print("Downloaded remote Prism2 artifacts.")

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


def calculate_sha256(path: str | Path) -> str:
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        while chunk := file.read(1024 * 1024):
            digest.update(chunk)

    return digest.hexdigest()


remote_manifest = pd.read_csv(
    remote_manifest_path
)

with open(
    remote_metadata_path,
    encoding="utf-8",
) as metadata_file:
    remote_metadata = json.load(metadata_file)

with np.load(
    remote_embedding_path,
    allow_pickle=False,
) as cache:
    remote_wsi_ids = (
        cache["wsi_ids"].astype(str)
    )
    remote_embeddings = (
        cache["embeddings"]
    )
    remote_tile_counts = (
        cache["tile_counts"]
    )

    if remote_embeddings.shape != (
        204,
        2560,
    ):
        raise RuntimeError(
            "Unexpected embedding shape: "
            f"{remote_embeddings.shape}"
        )

    if remote_embeddings.dtype != np.float16:
        raise RuntimeError(
            "Unexpected embedding dtype: "
            f"{remote_embeddings.dtype}"
        )

    if not np.isfinite(
        remote_embeddings
    ).all():
        raise RuntimeError(
            "Remote embeddings contain "
            "NaN or infinity."
        )

    if len(set(remote_wsi_ids)) != 204:
        raise RuntimeError(
            "Remote WSI IDs are not unique."
        )

    manifest_wsi_ids = (
        remote_manifest["wsi_id"]
        .astype(str)
        .to_numpy()
    )

    if not np.array_equal(
        remote_wsi_ids,
        manifest_wsi_ids,
    ):
        raise RuntimeError(
            "NPZ WSI order does not match "
            "manifest order."
        )

    manifest_tile_counts = (
        remote_manifest["tile_count"]
        .to_numpy(dtype=np.int32)
    )

    if not np.array_equal(
        remote_tile_counts,
        manifest_tile_counts,
    ):
        raise RuntimeError(
            "NPZ tile counts do not match "
            "manifest."
        )

    expected_wsi_ids = set(
        tile_index["wsi_id"].astype(str)
    )

    observed_wsi_ids = set(
        remote_wsi_ids.tolist()
    )

    missing_wsi_ids = (
        expected_wsi_ids
        - observed_wsi_ids
    )
    extra_wsi_ids = (
        observed_wsi_ids
        - expected_wsi_ids
    )

    norms = np.linalg.norm(
        remote_embeddings.astype(
            np.float32
        ),
        axis=1,
    )

observed_sha256 = calculate_sha256(
    remote_embedding_path
)

expected_sha256 = remote_metadata[
    "embedding_sha256"
]

if observed_sha256 != expected_sha256:
    raise RuntimeError(
        "Remote NPZ SHA-256 mismatch."
    )

if int(remote_tile_counts.sum()) != 200488:
    raise RuntimeError(
        "Unexpected total tile count: "
        f"{remote_tile_counts.sum()}"
    )

if missing_wsi_ids:
    raise RuntimeError(
        f"Missing {len(missing_wsi_ids)} WSIs"
    )

if extra_wsi_ids:
    raise RuntimeError(
        f"Found {len(extra_wsi_ids)} "
        "unexpected WSIs"
    )

print("=== Prism2 remote validation ===")
print(
    "Embedding shape:",
    remote_embeddings.shape,
)
print(
    "Embedding dtype:",
    remote_embeddings.dtype,
)
print(
    "All finite:",
    bool(
        np.isfinite(
            remote_embeddings
        ).all()
    ),
)
print(
    "Unique WSI IDs:",
    len(observed_wsi_ids),
)
print(
    "Missing WSI IDs:",
    len(missing_wsi_ids),
)
print(
    "Extra WSI IDs:",
    len(extra_wsi_ids),
)
print(
    "Total input tiles:",
    int(remote_tile_counts.sum()),
)
print(
    "WSI order matches manifest:",
    True,
)
print(
    "Tile counts match manifest:",
    True,
)
print(
    "SHA-256 matches metadata:",
    observed_sha256 == expected_sha256,
)
print(
    "Mean L2 norm:",
    round(float(norms.mean()), 4),
)
print(
    "Min L2 norm:",
    round(float(norms.min()), 4),
)
print(
    "Max L2 norm:",
    round(float(norms.max()), 4),
)

In [ ]:
validation_report = {
    "artifact_revision_validated": (
        PRISM2_CACHE_REVISION
    ),
    "slide_count": 204,
    "embedding_dimension": 2560,
    "embedding_dtype": "float16",
    "all_finite": True,
    "unique_wsi_ids": 204,
    "missing_wsi_ids": 0,
    "extra_wsi_ids": 0,
    "total_input_tiles": 200488,
    "wsi_order_matches_manifest": True,
    "tile_counts_match_manifest": True,
    "sha256_matches_metadata": True,
    "mean_l2_norm": float(norms.mean()),
    "min_l2_norm": float(norms.min()),
    "max_l2_norm": float(norms.max()),
}

validation_path = (
    PRISM2_OUTPUT_DIR
    / "validation.json"
)

validation_path.write_text(
    json.dumps(
        validation_report,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

api.upload_file(
    repo_id=EMBEDDING_REPO,
    repo_type="dataset",
    path_or_fileobj=str(validation_path),
    path_in_repo="prism2/validation.json",
    commit_message=(
        "Add Prism2 validation report"
    ),
)

final_repo_info = api.repo_info(
    EMBEDDING_REPO,
    repo_type="dataset",
)

print(
    "Validation uploaded:",
    "prism2/validation.json",
)
print(
    "Final embedding repository revision:",
    final_repo_info.sha,
)